In [ ]:
!nvidia-smi

: 

In [1]:
!pip install -q edge-tts librosa soundfile jiwer

In [2]:
!nvidia-smi

from google.colab import drive
drive.mount("/content/drive")

Wed Nov 25 13:43:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

MessageError: Failed to issue request POST https://colab.pa.googleapis.com/v1/credential-propagation:enable: Bad Request
Response body: {
  "error": {
    "code": 400,
    "message": "Precondition check failed.",
    "status": "FAILED_PRECONDITION"
  }
}


In [ ]:
!pip install -q transformers datasets accelerate evaluate jiwer librosa soundfile torchaudio

from datasets import load_dataset

ds = load_dataset("ademax/vivos-vie-speech2text")
print(ds)

TEXT_COL = "raw_transcription"

def prep_vivos_hf(batch):
    return {
        "audio": batch["audio"],
        "sentence": (batch[TEXT_COL] or "").strip().lower(),
        "source": "vivos_hf",
    }

vivos_train = ds["train"].map(prep_vivos_hf, remove_columns=ds["train"].column_names, num_proc=2)
vivos_test = ds["test"].map(prep_vivos_hf, remove_columns=ds["test"].column_names, num_proc=2)

print("Train:", len(vivos_train), "| Test:", len(vivos_test))
print("Mẫu thử:", vivos_train[0]["sentence"], "| SR:", vivos_train[0]["audio"]["sampling_rate"])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/566 [00:00<?, ?B/s]

data/train-00000-of-00004-ded4cd66da578d(…): reconstructing file:   0%|          |  0.00B /  414MB            

data/train-00000-of-00004-ded4cd66da578d(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-c9714e3bcd9a48(…): reconstructing file:   0%|          |  0.00B /  404MB            

data/train-00001-of-00004-c9714e3bcd9a48(…): downloading bytes:           |  0.00B            

... (8 outputs left out)

DatasetDict({
    train: Dataset({
        features: ['audio', 'transcription', 'raw_transcription'],
        num_rows: 11420
    })
    test: Dataset({
        features: ['audio', 'transcription', 'raw_transcription'],
        num_rows: 1000
    })
})


Map (num_proc=2):   0%|          | 0/11420 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 11420 | Test: 1000
Mẫu thử: ba mươi sáu ba mươi bảy | SR: 16000


In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

MODEL_ID = "vinai/PhoWhisper-small"

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language="Vietnamese", task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_ID, language="Vietnamese", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

model.generation_config.language = "vietnamese"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

encoded_train = vivos_train.map(prepare_dataset, remove_columns=vivos_train.column_names, num_proc=2)
encoded_eval = vivos_test.map(prepare_dataset, remove_columns=vivos_test.column_names, num_proc=2)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/asr_project/phowhisper-small-vi",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=20,
    max_steps=100,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=50,
    eval_steps=50,
    logging_steps=20,
    report_to=["none"],
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

trainer.train()

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

... (10 outputs left out)

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=100, training_loss=0.2397229093313217, metrics={'train_runtime': 1309.0387, 'train_samples_per_second': 1.222, 'train_steps_per_second': 0.076, 'total_flos': 4.61736640512e+17, 'train_loss': 0.2397229093313217, 'epoch': 0.1400560224089636})

In [ ]:
training_args.max_steps = 400
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
trainer.train(resume_from_checkpoint=True)

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Step,Training Loss,Validation Loss,Wer
150,0.059857,0.035596,2.869112
200,0.045831,0.032834,1.754663
250,0.048908,0.033758,1.580778
300,0.056687,0.031857,2.734745
350,0.062983,0.031227,1.517547
400,0.032853,0.030776,1.881126


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=400, training_loss=0.05525294005870819, metrics={'train_runtime': 4103.0595, 'train_samples_per_second': 1.17, 'train_steps_per_second': 0.073, 'total_flos': 1.846946562048e+18, 'train_loss': 0.05525294005870819, 'epoch': 0.5602240896358543})

In [ ]:
BASE_CKPT_DIR = "/content/drive/MyDrive/asr_project/stage1_vivos_step100"
model.save_pretrained(BASE_CKPT_DIR)
processor.save_pretrained(BASE_CKPT_DIR)
print("Đã lưu xong checkpoint giai đoạn 1 tại:", BASE_CKPT_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu xong checkpoint giai đoạn 1 tại: /content/drive/MyDrive/asr_project/stage1_vivos_step100


In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount("/content/drive")

!ls -lh /content/drive/MyDrive/asr_project/stage1_vivos_step100

Tue Nov 24 17:33:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!ls -la /content/drive/MyDrive
!find /content/drive/MyDrive -maxdepth 3 -iname "*stage1*" -o -iname "*asr_project*"

total 4121
drwx------ 4 root root    4096 Nov 24 17:29  .
drwx------ 6 root root    4096 Nov 24 17:29  ..
-rw------- 1 root root     187 Nov 30  2025 '1. Bài đặt câu hỏi số 1 (tham khảo).gdoc'
-rw------- 1 root root  829432 Oct 10  2025 '3. Exercise.pdf'
drwx------ 2 root root    4096 May 13  2025 'Ảnh demo đồ án cơ sở 3'
drwx------ 3 root root    4096 Nov 24 17:24  asr_project
-rw------- 1 root root     187 Dec  4  2025 'Bảng chấm công T12 2025.gsheet'
-rw------- 1 root root  350377 Jul 31 10:24  BangchamcongT7-2026.xlsx
-rw------- 1 root root  415288 Sep  1 15:28  BangchamcongT8-2026.xlsx
-rw------- 1 root root  423182 Oct  6 12:21  BangchamcongT9-2026.xlsx
-rw------- 1 root root     187 May 23  2025 'Bảng tính không có tiêu đề (1).gsheet'
-rw------- 1 root root     187 May 23  2025 'Bảng tính không có tiêu đề.gsheet'
-rw------- 1 root root    3349 Aug 15  2025 'Biểu đồ không có tiêu đề.drawio'
-rw------- 1 root root     187 Nov 23 14:46 'CF 

In [ ]:
!ls -la /content/drive/MyDrive/asr_project
!find /content/drive/MyDrive/asr_project -maxdepth 3

total 12
drwx------ 3 root root 4096 Nov 24 17:24 .
drwx------ 4 root root 4096 Nov 24 17:29 ..
drwx------ 5 root root 4096 Nov 24 17:24 phowhisper-small-vi
/content/drive/MyDrive/asr_project
/content/drive/MyDrive/asr_project/phowhisper-small-vi
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-150
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-150/optimizer.pt
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-150/rng_state.pth
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-150/generation_config.json
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-100
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-100/preprocessor_config.json
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-100/scaler.pt
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-100/scheduler.pt
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-200
/content/drive/MyDrive/as

In [ ]:
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-200

total 7.0K
-rw------- 1 root root 1.3K Nov 24 17:23 config.json
-rw------- 1 root root 5.3K Nov 24 17:23 training_args.bin


In [ ]:
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-150
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-100
!find /content/drive/MyDrive/asr_project -type f \( -name "*.safetensors" -o -name "*.bin" \)

total 1.9G
-rw------- 1 root root 4.5K Nov 24 17:23 generation_config.json
-rw------- 1 root root 1.9G Nov 24 17:23 optimizer.pt
-rw------- 1 root root  15K Nov 24 17:23 rng_state.pth
total 3.5K
-rw------- 1 root root  315 Nov 24 17:23 preprocessor_config.json
-rw------- 1 root root 1.4K Nov 24 17:23 scaler.pt
-rw------- 1 root root 1.5K Nov 24 17:23 scheduler.pt
/content/drive/MyDrive/asr_project/phowhisper-small-vi/checkpoint-200/training_args.bin


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

MODEL_ID = "vinai/PhoWhisper-small"
processor = WhisperProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.generation_config.language = "vietnamese"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

print("Đã nạp xong:", MODEL_ID)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

... (5 outputs left out)

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Đã nạp xong: vinai/PhoWhisper-small


In [ ]:
from datasets import load_dataset, Audio, concatenate_datasets

# 1. FLEURS tiếng Việt
fleurs_train_raw = load_dataset("google/fleurs", "vi_vn", split="train", trust_remote_code=True)
fleurs_test_raw = load_dataset("google/fleurs", "vi_vn", split="test", trust_remote_code=True)

def prep_fleurs(batch):
    return {
        "audio": batch["audio"],
        "sentence": (batch["transcription"] or "").strip().lower(),
        "source": "fleurs",
    }

fleurs_train = (
    fleurs_train_raw.map(prep_fleurs, remove_columns=fleurs_train_raw.column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)
fleurs_test = (
    fleurs_test_raw.map(prep_fleurs, remove_columns=fleurs_test_raw.column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)

# 2. VIVOS
ds_vivos = load_dataset("ademax/vivos-vie-speech2text")

def prep_vivos(batch):
    return {
        "audio": batch["audio"],
        "sentence": (batch["raw_transcription"] or "").strip().lower(),
        "source": "vivos",
    }

vivos_train_sub = (
    ds_vivos["train"]
    .shuffle(seed=42)
    .select(range(2500))
    .map(prep_vivos, remove_columns=ds_vivos["train"].column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)
vivos_test_sub = (
    ds_vivos["test"]
    .shuffle(seed=42)
    .select(range(300))
    .map(prep_vivos, remove_columns=ds_vivos["test"].column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)

train_dataset_raw = concatenate_datasets([fleurs_train, vivos_train_sub]).shuffle(seed=42)
eval_dataset_raw = concatenate_datasets([
    fleurs_test.select(range(min(300, len(fleurs_test)))),
    vivos_test_sub,
])

print("Train samples:", len(train_dataset_raw), "| Eval samples:", len(eval_dataset_raw))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

parquet-data/vi_vn/train-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 2.07GB            

parquet-data/vi_vn/train-00000-of-00001.(…): downloading bytes:           |  0.00B            

parquet-data/vi_vn/validation-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  275MB            

parquet-data/vi_vn/validation-00000-of-0(…): downloading bytes:           |  0.00B            

... (20 outputs left out)

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/2500 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

Train samples: 5494 | Eval samples: 600


In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

encoded_train = train_dataset_raw.map(
    prepare_dataset, remove_columns=train_dataset_raw.column_names, num_proc=2
)
encoded_eval = eval_dataset_raw.map(
    prepare_dataset, remove_columns=eval_dataset_raw.column_names, num_proc=2
)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/asr_project/phowhisper-small-vivos-fleurs",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=30,
    max_steps=250,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=50,
    eval_steps=50,
    logging_steps=25,
    report_to=["none"],
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

trainer.train()

# Lưu bản model sạch hoàn chỉnh sau bước này để làm nền fine-tune giọng trẻ em 5-7 tuổi
model.save_pretrained("/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base")
processor.save_pretrained("/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base")

ModuleNotFoundError: No module named 'evaluate'

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!ls -lh /content/drive/MyDrive/asr_project/phowhisper-small-vivos-fleurs
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base

Mounted at /content/drive
total 8.0K
drwx------ 2 root root 4.0K Nov 24 18:06 checkpoint-100
drwx------ 2 root root 4.0K Nov 24 17:56 checkpoint-50
ls: cannot access '/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base': No such file or directory


In [ ]:
!pip install -q evaluate jiwer

import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

encoded_train = train_dataset_raw.map(
    prepare_dataset, remove_columns=train_dataset_raw.column_names, num_proc=2
)
encoded_eval = eval_dataset_raw.map(
    prepare_dataset, remove_columns=eval_dataset_raw.column_names, num_proc=2
)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/asr_project/phowhisper-small-vivos-fleurs",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=30,
    max_steps=250,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=50,
    eval_steps=50,
    logging_steps=25,
    report_to=["none"],
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

trainer.train()

model.save_pretrained("/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base")
processor.save_pretrained("/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base")
print("Đã lưu xong bản nền VIVOS + FLEURS tại: /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base")

Map (num_proc=2):   0%|          | 0/5494 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/600 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Wer
50,1.432881,0.660080,18.379431


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

CKPT_PATH = "/content/drive/MyDrive/asr_project/phowhisper-small-vivos-fleurs/checkpoint-100"
processor = WhisperProcessor.from_pretrained("vinai/PhoWhisper-small")
tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor
model = WhisperForConditionalGeneration.from_pretrained(CKPT_PATH)
model.generation_config.language = "vietnamese"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.20M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!ls -lh /content/drive/MyDrive/asr_project/phowhisper-small-vivos-fleurs
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 8.0K
drwx------ 2 root root 4.0K Nov 24 18:06 checkpoint-100
drwx------ 2 root root 4.0K Nov 24 17:56 checkpoint-50
ls: cannot access '/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base': No such file or directory


In [ ]:
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-small-vivos-fleurs/checkpoint-100
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-small-vivos-fleurs/checkpoint-50

total 2.8G
-rw------- 1 root root 1.3K Nov 24 18:06 config.json
-rw------- 1 root root 4.5K Nov 24 18:06 generation_config.json
-rw------- 1 root root 923M Nov 24 18:06 model.safetensors
-rw------- 1 root root 1.9G Nov 24 18:06 optimizer.pt
-rw------- 1 root root  315 Nov 24 18:06 preprocessor_config.json
-rw------- 1 root root  15K Nov 24 18:07 rng_state.pth
-rw------- 1 root root 1.4K Nov 24 18:07 scaler.pt
-rw------- 1 root root 1.5K Nov 24 18:07 scheduler.pt
-rw------- 1 root root 2.0K Nov 24 18:07 trainer_state.json
-rw------- 1 root root 5.3K Nov 24 18:06 training_args.bin
total 2.8G
-rw------- 1 root root 1.3K Nov 24 17:56 config.json
-rw------- 1 root root 4.5K Nov 24 17:56 generation_config.json
-rw------- 1 root root 923M Nov 24 17:57 model.safetensors
-rw------- 1 root root 1.9G Nov 24 17:57 optimizer.pt
-rw------- 1 root root  315 Nov 24 17:57 preprocessor_config.json
-rw------- 1 root root  15K Nov 24 17:57 rng_state.pth
-rw------- 1 root root 1.4K Nov 24 17:57 scaler.pt
-

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import evaluate
from datasets import load_dataset, Audio, concatenate_datasets
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("vinai/PhoWhisper-small")
tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor

# 1. FLEURS tiếng Việt
fleurs_train_raw = load_dataset("google/fleurs", "vi_vn", split="train")
fleurs_test_raw = load_dataset("google/fleurs", "vi_vn", split="test")

def prep_fleurs(batch):
    return {
        "audio": batch["audio"],
        "sentence": (batch["transcription"] or "").strip().lower(),
        "source": "fleurs",
    }

fleurs_train = (
    fleurs_train_raw.map(prep_fleurs, remove_columns=fleurs_train_raw.column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)
fleurs_test = (
    fleurs_test_raw.map(prep_fleurs, remove_columns=fleurs_test_raw.column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)

# 2. VIVOS
ds_vivos = load_dataset("ademax/vivos-vie-speech2text")

def prep_vivos(batch):
    return {
        "audio": batch["audio"],
        "sentence": (batch["raw_transcription"] or "").strip().lower(),
        "source": "vivos",
    }

vivos_train_sub = (
    ds_vivos["train"]
    .shuffle(seed=42)
    .select(range(2500))
    .map(prep_vivos, remove_columns=ds_vivos["train"].column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)
vivos_test_sub = (
    ds_vivos["test"]
    .shuffle(seed=42)
    .select(range(300))
    .map(prep_vivos, remove_columns=ds_vivos["test"].column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)

train_dataset_raw = concatenate_datasets([fleurs_train, vivos_train_sub]).shuffle(seed=42)
eval_dataset_raw = concatenate_datasets([
    fleurs_test.select(range(min(300, len(fleurs_test)))),
    vivos_test_sub,
])

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

encoded_train = train_dataset_raw.map(
    prepare_dataset, remove_columns=train_dataset_raw.column_names, num_proc=2
)
encoded_eval = eval_dataset_raw.map(
    prepare_dataset, remove_columns=eval_dataset_raw.column_names, num_proc=2
)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

wer_metric = evaluate.load("wer")

ModuleNotFoundError: No module named 'evaluate'

In [ ]:
!pip install -q evaluate jiwer

In [ ]:
import jiwer

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100.0 * jiwer.wer(label_str, pred_str)}

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import jiwer
from datasets import load_dataset, Audio, concatenate_datasets
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("vinai/PhoWhisper-small")
tokenizer = processor.tokenizer
feature_extractor = processor.feature_extractor

# 1. FLEURS tiếng Việt
fleurs_train_raw = load_dataset("google/fleurs", "vi_vn", split="train")
fleurs_test_raw = load_dataset("google/fleurs", "vi_vn", split="test")

def prep_fleurs(batch):
    return {
        "audio": batch["audio"],
        "sentence": (batch["transcription"] or "").strip().lower(),
        "source": "fleurs",
    }

fleurs_train = (
    fleurs_train_raw.map(prep_fleurs, remove_columns=fleurs_train_raw.column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)
fleurs_test = (
    fleurs_test_raw.map(prep_fleurs, remove_columns=fleurs_test_raw.column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)

# 2. VIVOS
ds_vivos = load_dataset("ademax/vivos-vie-speech2text")

def prep_vivos(batch):
    return {
        "audio": batch["audio"],
        "sentence": (batch["raw_transcription"] or "").strip().lower(),
        "source": "vivos",
    }

vivos_train_sub = (
    ds_vivos["train"]
    .shuffle(seed=42)
    .select(range(2500))
    .map(prep_vivos, remove_columns=ds_vivos["train"].column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)
vivos_test_sub = (
    ds_vivos["test"]
    .shuffle(seed=42)
    .select(range(300))
    .map(prep_vivos, remove_columns=ds_vivos["test"].column_names, num_proc=2)
    .cast_column("audio", Audio(sampling_rate=16000))
)

train_dataset_raw = concatenate_datasets([fleurs_train, vivos_train_sub]).shuffle(seed=42)
eval_dataset_raw = concatenate_datasets([
    fleurs_test.select(range(min(300, len(fleurs_test)))),
    vivos_test_sub,
])

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

encoded_train = train_dataset_raw.map(
    prepare_dataset, remove_columns=train_dataset_raw.column_names, num_proc=2
)
encoded_eval = eval_dataset_raw.map(
    prepare_dataset, remove_columns=eval_dataset_raw.column_names, num_proc=2
)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

print("Xong encode dữ liệu! Train:", len(encoded_train), "| Eval:", len(encoded_eval))

README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

parquet-data/vi_vn/train-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 2.07GB            

parquet-data/vi_vn/train-00000-of-00001.(…): downloading bytes:           |  0.00B            

parquet-data/vi_vn/validation-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  275MB            

parquet-data/vi_vn/validation-00000-of-0(…): downloading bytes:           |  0.00B            

parquet-data/vi_vn/test-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B /  691MB            

... (20 outputs left out)

Map (num_proc=2):   0%|          | 0/300 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/5494 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/600 [00:00<?, ? examples/s]

Xong encode dữ liệu! Train: 5494 | Eval: 600


In [ ]:
from transformers import (
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

OUTPUT_DIR = "/content/drive/MyDrive/asr_project/phowhisper-small-vivos-fleurs"
CKPT_100 = f"{OUTPUT_DIR}/checkpoint-100"
SAVE_BASE_DIR = "/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base"

model = WhisperForConditionalGeneration.from_pretrained(CKPT_100)
model.generation_config.language = "vietnamese"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": 100.0 * jiwer.wer(label_str, pred_str)}

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=30,
    max_steps=250,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=50,
    eval_steps=50,
    logging_steps=25,
    report_to=["none"],
    save_total_limit=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=encoded_train,
    eval_dataset=encoded_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)

trainer.train(resume_from_checkpoint=CKPT_100)

model.save_pretrained(SAVE_BASE_DIR)
processor.save_pretrained(SAVE_BASE_DIR)
print("Đã hoàn tất 250 steps và lưu bản nền tại:", SAVE_BASE_DIR)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


Step,Training Loss,Validation Loss,Wer
150,0.723982,0.374818,15.013635
200,0.628060,0.323869,14.561745
250,0.582970,0.314428,15.239579


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã hoàn tất 250 steps và lưu bản nền tại: /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!ls -lh /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 926M
-rw------- 1 root root 1.3K Nov 25 05:33 config.json
-rw------- 1 root root 4.5K Nov 25 05:33 generation_config.json
-rw------- 1 root root 923M Nov 25 05:34 model.safetensors
-rw------- 1 root root  409 Nov 25 05:34 processor_config.json
-rw------- 1 root root 2.1K Nov 25 05:34 tokenizer_config.json
-rw------- 1 root root 3.5M Nov 25 05:34 tokenizer.json


In [5]:
import json
import asyncio
import random
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa
import pandas as pd
import edge_tts

CURRICULUM_PATH = "/content/drive/MyDrive/asr_project/curriculum.json"
OUT_DIR = Path("/content/child_synth_data")
WAV_DIR = OUT_DIR / "wavs"
WAV_DIR.mkdir(parents=True, exist_ok=True)

with open(CURRICULUM_PATH, "r", encoding="utf-8") as f:
    curr = json.load(f)

# Thu thập danh sách từ / âm tiết mục tiêu từ curriculum.json
target_texts = []
for lesson in curr.get("lessons", []):
    target_texts.extend(lesson.get("sample_syllables", []))
    target_texts.extend(lesson.get("combine_words", []))

# Bổ sung thêm các cặp từ dễ nhầm L/N, hỏi/ngã, nuốt âm cuối để tăng độ phủ lớp 1
extra_words = [
    "lúa", "núa", "làm", "nằm", "lòng", "nòng", "lên", "nên",
    "bể", "bễ", "củ", "cũ", "thỏ", "thõ", "mỏ", "mõ",
    "hát", "há", "mát", "má", "bát", "bá", "xanh", "xa", "cành", "cà"
]
texts = sorted(list({t.strip().lower() for t in (target_texts + extra_words) if t.strip()}))
print(f"Tổng số từ/âm tiết gốc: {len(texts)}")

# 1. Sinh audio TTS tiếng Việt (giọng nữ/nam chuẩn)
VOICES = ["vi-VN-HoaiMyNeural", "vi-VN-NamMinhNeural"]

async def synth_all():
    raw_items = []
    for idx, text in enumerate(texts):
        for v_idx, voice in enumerate(VOICES):
            raw_path = WAV_DIR / f"raw_{idx:03d}_v{v_idx}.mp3"
            wav_16k = WAV_DIR / f"base_{idx:03d}_v{v_idx}.wav"
            communicate = edge_tts.Communicate(text=text, voice=voice, rate="+0%")
            await communicate.save(str(raw_path))

            y, sr = librosa.load(str(raw_path), sr=16000, mono=True)
            sf.write(str(wav_16k), y, 16000)
            raw_path.unlink(missing_ok=True)

            raw_items.append({
                "base_id": f"s{idx:03d}_v{v_idx}",
                "wav_path": str(wav_16k),
                "sentence": text,
            })
    return raw_items

base_items = await synth_all()

# 2. Tạo các biến thể giả lập đặc trưng giọng trẻ 5-7 tuổi (nâng pitch + đổi tốc độ nhẹ + noise rất nhỏ)
random.seed(42)
np.random.seed(42)

rows = []
for item in base_items:
    y, sr = librosa.load(item["wav_path"], sr=16000, mono=True)
    # Giữ bản gốc
    rows.append({
        "audio_path": item["wav_path"],
        "sentence": item["sentence"],
        "speaker_group": item["base_id"],
        "aug_type": "clean",
    })

    # Tạo 3 biến thể "giọng trẻ" giả lập: nâng cao độ +3.5, +5.0, +6.5 semitones
    for k, n_steps in enumerate([3.5, 5.0, 6.5]):
        rate_factor = random.uniform(0.92, 1.12)
        y_shift = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)
        y_stretch = librosa.effects.time_stretch(y=y_shift, rate=rate_factor)
        # Thêm tạp âm nền nhẹ mức thấp để mô phỏng mic thực tế
        noise = np.random.normal(0, 0.002, size=len(y_stretch))
        y_final = np.clip(y_stretch + noise, -1.0, 1.0)

        aug_path = WAV_DIR / f"{item['base_id']}_child_aug{k}.wav"
        sf.write(str(aug_path), y_final, 16000)
        rows.append({
            "audio_path": str(aug_path),
            "sentence": item["sentence"],
            "speaker_group": item["base_id"],
            "aug_type": f"pitch_{n_steps}",
        })

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "metadata.csv", index=False, encoding="utf-8")
print("Đã tạo metadata.csv với tổng số mẫu:", len(df))

Tổng số từ/âm tiết gốc: 45
Đã tạo metadata.csv với tổng số mẫu: 360


In [4]:
import json
from pathlib import Path

CURRICULUM_PATH = Path("/content/drive/MyDrive/asr_project/curriculum.json")
CURRICULUM_PATH.parent.mkdir(parents=True, exist_ok=True)

curriculum_data = {
    "grade": 1,
    "curriculum_version": "1.0.0",
    "learning_flow": ["listen", "speak", "write", "combine"],
    "mouth_guides": {
        "mouth_l_tip_ridge": {
            "title": "Âm đầu L — Đầu lưỡi chạm lợi trên",
            "instruction": "Nâng đầu lưỡi chạm nhẹ vào phần lợi ngay sau răng cửa trên, để luồng hơi thoát hai bên cạnh lưỡi, rung thanh quản rõ.",
            "image_key": "mouth_l.png"
        },
        "mouth_n_nasal": {
            "title": "Âm đầu N — Lưỡi chạm lợi & hơi lên mũi",
            "instruction": "Đầu lưỡi áp sát lợi trên chặn luồng hơi miệng, hạ màn hầu để hơi vang nhẹ qua mũi.",
            "image_key": "mouth_n.png"
        },
        "mouth_tone_hoi": {
            "title": "Thanh hỏi — Giọng hạ thấp rồi hơi nâng nhẹ",
            "instruction": "Đọc giọng trầm hơn mức bình thường, hạ thấp rồi nâng nhẹ đều, không ngắt giọng đột ngột hay bật hơi gằn.",
            "image_key": "tone_hoi.png"
        },
        "mouth_tone_nga": {
            "title": "Thanh ngã — Giọng trầm, ngắt nhẹ rồi lướt lên",
            "instruction": "Hạ giọng, ngắt hơi nhẹ ở giữa rồi lướt cao lên rõ ràng, phân biệt dứt khoát với thanh hỏi.",
            "image_key": "tone_nga.png"
        },
        "mouth_final_stop": {
            "title": "Âm cuối (-p, -t, -c, -ch) — Khép môi hoặc chặn lưỡi cuối âm",
            "instruction": "Không dừng ở nguyên âm: khép chặt hai môi (với -p) hoặc giữ đầu lưỡi/gốc lưỡi chạm sát vị trí kết thúc (-t, -c, -ch) để khép gọn âm tiết.",
            "image_key": "final_stop.png"
        },
        "mouth_final_nasal": {
            "title": "Âm cuối (-m, -n, -ng) — Giữ âm vang mũi ở cuối vần",
            "instruction": "Kéo dài nhẹ phần vang ở cuối từ trước khi khép môi (-m), chạm lợi (-n) hoặc nâng gốc lưỡi (-ng), không nuốt mất đuôi từ.",
            "image_key": "final_nasal.png"
        }
    },
    "lessons": [
        {
            "lesson_id": "tv1_u01_l_ba",
            "unit": 1,
            "title": "Âm l, vần a và tiếng ba / lá",
            "target_phoneme": "l",
            "target_rime": "a",
            "spelling_guide": "lờ - a - la - hỏi - lả | bờ - a - ba",
            "sample_syllables": ["la", "lả", "lã", "ba", "bà"],
            "combine_words": ["lá", "lả", "ba", "bà", "la cà"],
            "common_errors": ["l_n_confusion", "hoi_nga_confusion", "final_consonant_drop"],
            "mouth_guide_ids": ["mouth_l_tip_ridge", "mouth_tone_hoi", "mouth_tone_nga"],
            "writing_module": {
                "writing_lesson_id": "write_tv1_l",
                "letters_to_trace": ["l", "a", "b"],
                "combine_word_to_write": "lá"
            }
        },
        {
            "lesson_id": "tv1_u02_n_na",
            "unit": 2,
            "title": "Âm n, thanh ngã và tiếng nã / nả",
            "target_phoneme": "n",
            "target_rime": "a",
            "spelling_guide": "nờ - a - na - ngã - nã | nờ - a - na - hỏi - nả",
            "sample_syllables": ["na", "nả", "nã", "nơ", "nở"],
            "combine_words": ["nả", "nã", "no", "nở", "nơ"],
            "common_errors": ["l_n_confusion", "hoi_nga_confusion"],
            "mouth_guide_ids": ["mouth_n_nasal", "mouth_tone_nga", "mouth_tone_hoi"],
            "writing_module": {
                "writing_lesson_id": "write_tv1_n",
                "letters_to_trace": ["n", "a"],
                "combine_word_to_write": "nã"
            }
        },
        {
            "lesson_id": "tv1_u03_final_stop_at",
            "unit": 3,
            "title": "Vần at và âm cuối -t",
            "target_phoneme": "t",
            "target_rime": "at",
            "spelling_guide": "a - tờ - át | hát",
            "sample_syllables": ["át", "hát", "mát", "bát"],
            "combine_words": ["hát", "mát", "bát", "cát"],
            "common_errors": ["final_consonant_drop"],
            "mouth_guide_ids": ["mouth_final_stop"],
            "writing_module": {
                "writing_lesson_id": "write_tv1_at",
                "letters_to_trace": ["a", "t"],
                "combine_word_to_write": "hát"
            }
        },
        {
            "lesson_id": "tv1_u04_final_nasal_ang",
            "unit": 4,
            "title": "Vần ang và âm cuối -ng",
            "target_phoneme": "ng",
            "target_rime": "ang",
            "spelling_guide": "a - ngờ - ang | xanh",
            "sample_syllables": ["ang", "xanh", "chành", "bảng"],
            "combine_words": ["xanh", "bảng", "cành", "mành"],
            "common_errors": ["final_consonant_drop", "hoi_nga_confusion"],
            "mouth_guide_ids": ["mouth_final_nasal", "mouth_tone_hoi"],
            "writing_module": {
                "writing_lesson_id": "write_tv1_ang",
                "letters_to_trace": ["a", "n", "g"],
                "combine_word_to_write": "xanh"
            }
        }
    ]
}

with open(CURRICULUM_PATH, "w", encoding="utf-8") as f:
    json.dump(curriculum_data, f, ensure_ascii=False, indent=2)

print("Đã tạo thành công:", CURRICULUM_PATH)

Đã tạo thành công: /content/drive/MyDrive/asr_project/curriculum.json


In [6]:
import pandas as pd
from datasets import Dataset, Audio

df = pd.read_csv("/content/child_synth_data/metadata.csv")
unique_groups = df["speaker_group"].unique().tolist()
random.Random(42).shuffle(unique_groups)

split_idx = int(0.85 * len(unique_groups))
train_groups = set(unique_groups[:split_idx])
eval_groups = set(unique_groups[split_idx:])

train_df = df[df["speaker_group"].isin(train_groups)].reset_index(drop=True)
eval_df = df[df["speaker_group"].isin(eval_groups)].reset_index(drop=True)

child_train_ds = Dataset.from_pandas(train_df).cast_column("audio_path", Audio(sampling_rate=16000)).rename_column("audio_path", "audio")
child_eval_ds = Dataset.from_pandas(eval_df).cast_column("audio_path", Audio(sampling_rate=16000)).rename_column("audio_path", "audio")

print(f"Train mẫu: {len(child_train_ds)} | Eval mẫu: {len(child_eval_ds)}")

Train mẫu: 304 | Eval mẫu: 56


In [7]:
import torch
import jiwer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

BASE_MODEL_DIR = "/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = WhisperProcessor.from_pretrained(BASE_MODEL_DIR)
base_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_DIR).to(device)
base_model.eval()
base_model.generation_config.language = "vietnamese"
base_model.generation_config.task = "transcribe"
base_model.generation_config.forced_decoder_ids = None

def evaluate_asr(mdl, ds):
    refs, hyps = [], []
    mdl.eval()
    with torch.no_grad():
        for item in ds:
            audio = item["audio"]
            inp = processor.feature_extractor(
                audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt"
            ).input_features.to(device)
            if device == "cuda":
                inp = inp.half()
                mdl = mdl.half()
            gen_ids = mdl.generate(inp, max_new_tokens=64)
            pred = processor.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0].strip().lower()
            refs.append(item["sentence"].strip().lower())
            hyps.append(pred)
    wer = 100.0 * jiwer.wer(refs, hyps)
    cer = 100.0 * jiwer.cer(refs, hyps)
    return wer, cer, refs, hyps

base_wer, base_cer, _, _ = evaluate_asr(base_model, child_eval_ds)
print(f"[BASELINE] WER: {base_wer:.2f}% | CER: {base_cer:.2f}%")

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base'. Use `repo_type` argument if needed.

In [10]:
import os
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

BASE_MODEL_DIR = "/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base"
print("Tồn tại thư mục:", os.path.isdir(BASE_MODEL_DIR))
!ls -la /content/drive/MyDrive/asr_project 2>/dev/null || echo "Chưa có thư mục asr_project trong MyDrive hiện tại"

Tồn tại thư mục: False
total 16
drwxr-xr-x 2 root root 4096 Nov 25 13:45 .
drwxr-xr-x 3 root root 4096 Nov 25 13:45 ..
-rw-r--r-- 1 root root 5409 Nov 25 13:45 curriculum.json


In [11]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

!ls -lh /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base

ValueError: Mountpoint must not already contain files

In [12]:
import os, shutil
from google.colab import drive

mount_dir = "/content/drive"
if os.path.exists(mount_dir):
    # Thử unmount trước nếu đang bị treo
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    # Xóa sạch nội dung bên trong /content/drive nếu vẫn còn file
    if os.path.isdir(mount_dir):
        for name in os.listdir(mount_dir):
            p = os.path.join(mount_dir, name)
            if os.path.islink(p) or os.path.isfile(p):
                os.unlink(p)
            elif os.path.isdir(p):
                shutil.rmtree(p, ignore_errors=True)

drive.mount(mount_dir)
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base

Mounted at /content/drive
ls: cannot access '/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base': No such file or directory


In [9]:
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base

ls: cannot access '/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base': No such file or directory


In [13]:
import os, shutil
from google.colab import drive

mount_dir = "/content/drive"
if os.path.exists(mount_dir):
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    if os.path.isdir(mount_dir):
        for name in os.listdir(mount_dir):
            p = os.path.join(mount_dir, name)
            if os.path.islink(p) or os.path.isfile(p):
                os.unlink(p)
            elif os.path.isdir(p):
                shutil.rmtree(p, ignore_errors=True)

drive.mount(mount_dir)
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base

Mounted at /content/drive
ls: cannot access '/content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base': No such file or directory


In [14]:
!ls -la /content/drive/MyDrive
!ls -la /content/drive/MyDrive/asr_project 2>/dev/null || echo "Chua co asr_project trong MyDrive"

ls: /content/drive/MyDrive/phowhisper-small-vi
ls: '/content/drive/MyDrive/phowhisper-small-vi (1)'
total 150052
drwx------ 32 root root     4096 Nov 25 15:02  .
drwx------  5 root root     4096 Nov 25 15:02  ..
-rw-------  1 root root  1772136 Apr  2  2026 '2026.03.25_Cảnh bảo học tập - Buộc thôi học_HK1 25-26 (1).xlsx'
-rw-------  1 root root  1772136 Apr  2  2026 '2026.03.25_Cảnh bảo học tập - Buộc thôi học_HK1 25-26.xlsx'
drwx------  2 root root     4096 Apr  9  2024 '23IT.B066 Nguyễn Văn Hoàng'
-rw-------  1 root root 39497667 Sep  2 19:53 '23IT.B066 - Nguyen Van Hoang - Bai Thuc Hanh 03.docx'
-rw-------  1 root root  3683084 Sep 10 02:32 '23IT.B066 - NguyenVanHoang - Bai Thuc Hanh 04.docx'
-rw-------  1 root root  3036500 Sep 17 16:23 '23IT.B066 - Nguyen Van Hoang - Bai Thuc Hanh W1.docx'
-rw-------  1 root root   286229 Oct 14 20:12 '23IT.B066 - NguyenVanHoang – BaiThuHoach.docx'
-rw-------  1 root root    64083 Nov 11 07:27 '23IT.B066 - NguyenVanHoang - Ba

In [15]:
!find /content/drive -maxdepth 4 -name "config.json" 2>/dev/null

In [16]:
!ls -lh /content/drive/MyDrive/asr_project/phowhisper-vivos-fleurs-base

total 926M
-rw------- 1 root root 1.3K Nov 25 05:33 config.json
-rw------- 1 root root 4.5K Nov 25 05:33 generation_config.json
-rw------- 1 root root 923M Nov 25 05:34 model.safetensors
-rw------- 1 root root  409 Nov 25 05:34 processor_config.json
-rw------- 1 root root 2.1K Nov 25 05:34 tokenizer_config.json
-rw------- 1 root root 3.5M Nov 25 05:34 tokenizer.json


In [17]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

BASE_MODEL_DIR = "/content/drive/MyDrive/phowhisper-small-vivos-fleurs"  # Thay bằng đúng đường dẫn thư mục vừa ls ra ở trên
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

processor = WhisperProcessor.from_pretrained(BASE_MODEL_DIR, local_files_only=True)
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_DIR,
    local_files_only=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
base_model.eval()

Using device: cuda


HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/phowhisper-small-vivos-fleurs'. Use `repo_type` argument if needed.

In [18]:
import os
import torch
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
)

BASE_MODEL_DIR = "/content/drive/MyDrive/phowhisper-small-vivos-fleurs"
device = "cuda" if torch.cuda.is_available() else "cpu"

print("is_dir:", os.path.isdir(BASE_MODEL_DIR))
print("files:", sorted(os.listdir(BASE_MODEL_DIR)))

# 1. Nạp model trọng số trực tiếp từ thư mục local
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_DIR,
    local_files_only=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
base_model.eval()
print("Loaded base_model OK")

# 2. Thử nạp tokenizer local; nếu thiếu file vocab/merges thì lấy từ mốc gốc Hugging Face
try:
    tokenizer = WhisperTokenizerFast.from_pretrained(BASE_MODEL_DIR, local_files_only=True)
    print("Loaded tokenizer local OK")
except Exception as e:
    print("Tokenizer local lỗi:", e)
    tokenizer = WhisperTokenizerFast.from_pretrained("vinai/PhoWhisper-small")

# 3. Thử nạp feature_extractor local; nếu thiếu preprocessor_config.json thì lấy từ mốc gốc
try:
    feature_extractor = WhisperFeatureExtractor.from_pretrained(BASE_MODEL_DIR, local_files_only=True)
    print("Loaded feature_extractor local OK")
except Exception as e:
    print("FeatureExtractor local lỗi:", e)
    feature_extractor = WhisperFeatureExtractor.from_pretrained("vinai/PhoWhisper-small")

is_dir: False


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/phowhisper-small-vivos-fleurs'

In [19]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

BASE_MODEL_DIR = "vinai/PhoWhisper-small"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

processor = WhisperProcessor.from_pretrained(BASE_MODEL_DIR)
base_model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
base_model.eval()

Using device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

... (6 outputs left out)

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
 

In [20]:
base_model.generation_config.language = "vi"
base_model.generation_config.task = "transcribe"
base_model.generation_config.forced_decoder_ids = None

In [21]:
SAVE_DIR = "/content/drive/MyDrive/phowhisper-child-5-7-final"
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

NameError: name 'model' is not defined

In [22]:
SAVE_DIR = "/content/drive/MyDrive/phowhisper-child-5-7-final"
trainer.model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

NameError: name 'trainer' is not defined

In [23]:
SAVE_DIR = "/content/drive/MyDrive/phowhisper-child-5-7-final"
trainer.model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

NameError: name 'trainer' is not defined

In [24]:
# Cài nhanh nếu môi trường chưa có jiwer / evaluate
# !pip install -q jiwer evaluate

import re
import torch
from jiwer import wer, cer

def normalize_vi(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\sàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def transcribe_one(audio_16k, sampling_rate=16000) -> str:
    inputs = processor(audio_16k, sampling_rate=sampling_rate, return_tensors="pt")
    input_features = inputs.input_features.to(
        device,
        dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    with torch.no_grad():
        pred_ids = base_model.generate(input_features)
    return processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

# Ví dụ chạy trên eval_samples (list các dict có "audio" và "text")
refs, hyps = [], []
for sample in eval_samples:
    ref_norm = normalize_vi(sample["text"])
    hyp_norm = normalize_vi(transcribe_one(sample["audio"]))
    refs.append(ref_norm)
    hyps.append(hyp_norm)

baseline_wer = wer(refs, hyps)
baseline_cer = cer(refs, hyps)
print(f"Baseline WER: {baseline_wer:.4f} ({baseline_wer*100:.2f}%)")
print(f"Baseline CER: {baseline_cer:.4f} ({baseline_cer*100:.2f}%)")

NameError: name 'eval_samples' is not defined

In [25]:
# !pip install -q gTTS librosa jiwer
import io
import librosa
from gtts import gTTS

test_texts = [
    "làng quê",
    "nàng tiên",
    "bé đọc bài",
    "mẹ nấu cơm",
    "con chim hót",
]

eval_samples = []
for txt in test_texts:
    buf = io.BytesIO()
    gTTS(text=txt, lang="vi").write_to_fp(buf)
    buf.seek(0)
    audio_16k, _ = librosa.load(buf, sr=16000, mono=True)
    eval_samples.append({"audio": audio_16k, "text": txt})

print("Đã tạo", len(eval_samples), "mẫu test nhanh")

ModuleNotFoundError: No module named 'gtts'

In [26]:
!pip install -q gTTS librosa jiwer

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.18 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [27]:
import gtts, librosa, jiwer
print("Import OK!")

Import OK!


In [28]:
!pip install -q "click>=8.4.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.5.1 which is incompatible.


In [29]:
import io, json, os, librosa
from gtts import gTTS

def extract_texts_from_curriculum(path="data/curriculum.json", limit=8):
    texts = []
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        def walk(obj):
            if len(texts) >= limit:
                return
            if isinstance(obj, str):
                s = obj.strip()
                if 1 <= len(s) <= 40 and s not in texts:
                    texts.append(s)
            elif isinstance(obj, dict):
                for v in obj.values():
                    walk(v)
            elif isinstance(obj, list):
                for item in obj:
                    walk(item)

        walk(data)
    return texts[:limit]

test_texts = extract_texts_from_curriculum("data/curriculum.json", limit=8)
if not test_texts:
    test_texts = ["làng quê", "nàng tiên", "bé đọc bài", "mẹ nấu cơm", "con chim hót"]

eval_samples = []
for txt in test_texts:
    buf = io.BytesIO()
    gTTS(text=txt, lang="vi").write_to_fp(buf)
    buf.seek(0)
    audio_16k, _ = librosa.load(buf, sr=16000, mono=True)
    eval_samples.append({"audio": audio_16k, "text": txt})

print("Mẫu test:", test_texts)
print("Số mẫu eval_samples:", len(eval_samples))

Mẫu test: ['làng quê', 'nàng tiên', 'bé đọc bài', 'mẹ nấu cơm', 'con chim hót']
Số mẫu eval_samples: 5


In [30]:
import re, torch
from jiwer import wer, cer

def normalize_vi(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\sàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def transcribe_one(audio_16k, sampling_rate=16000) -> str:
    inputs = processor(audio_16k, sampling_rate=sampling_rate, return_tensors="pt")
    input_features = inputs.input_features.to(
        device,
        dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    with torch.no_grad():
        pred_ids = base_model.generate(input_features)
    return processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

refs, hyps = [], []
for sample in eval_samples:
    ref_norm = normalize_vi(sample["text"])
    hyp_norm = normalize_vi(transcribe_one(sample["audio"]))
    refs.append(ref_norm)
    hyps.append(hyp_norm)
    print(f"REF: {ref_norm} | HYP: {hyp_norm}")

print(f"Baseline WER: {wer(refs, hyps):.4f} ({wer(refs, hyps)*100:.2f}%)")
print(f"Baseline CER: {cer(refs, hyps):.4f} ({cer(refs, hyps)*100:.2f}%)")

[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


REF: làng quê | HYP: làng quê
REF: nàng tiên | HYP: nàng tiên
REF: bé đọc bài | HYP: bé đọc bài
REF: mẹ nấu cơm | HYP: mẹ nấu cơm
REF: con chim hót | HYP: con chim hót
Baseline WER: 0.0000 (0.00%)
Baseline CER: 0.0000 (0.00%)


In [31]:
from datasets import load_dataset, Audio

# Ví dụ nạp subset tiếng Việt từ FLEURS (hoặc thay bằng dataset / file audio của bạn)
raw_train = load_dataset("google/fleurs", "vi_vn", split="train[:200]", trust_remote_code=True)
raw_eval = load_dataset("google/fleurs", "vi_vn", split="validation[:50]", trust_remote_code=True)

raw_train = raw_train.cast_column("audio", Audio(sampling_rate=16000))
raw_eval = raw_eval.cast_column("audio", Audio(sampling_rate=16000))

def prepare_batch(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcription"]).input_ids
    return batch

train_dataset = raw_train.map(prepare_batch, remove_columns=raw_train.column_names)
eval_dataset = raw_eval.map(prepare_batch, remove_columns=raw_eval.column_names)
print("Train samples:", len(train_dataset), "| Eval samples:", len(eval_dataset))

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/386k [00:00<?, ?B/s]

parquet-data/vi_vn/train-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 2.07GB            

parquet-data/vi_vn/train-00000-of-00001.(…): downloading bytes:           |  0.00B            

parquet-data/vi_vn/validation-00000-of-0(…): reconstructing file:   0%|          |  0.00B /  275MB            

parquet-data/vi_vn/validation-00000-of-0(…): downloading bytes:           |  0.00B            

... (5 outputs left out)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Train samples: 200 | Eval samples: 50


In [32]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# Nạp lại model float32 sạch để train ổn định trên GPU
from transformers import WhisperForConditionalGeneration
model_to_train = WhisperForConditionalGeneration.from_pretrained("vinai/PhoWhisper-small")
model_to_train.generation_config.language = "vi"
model_to_train.generation_config.task = "transcribe"
model_to_train.generation_config.forced_decoder_ids = None

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model_to_train.config.decoder_start_token_id,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/tmp_child_asr",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=20,
    num_train_epochs=2,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    predict_with_generate=True,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model_to_train,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    processing_class=processor.feature_extractor,
)

trainer.train()

Loading weights:   0%|          | 0/480 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,4.879873,1.232663
2,2.164342,0.816872


TrainOutput(global_step=26, training_loss=3.0944142708411584, metrics={'train_runtime': 111.9948, 'train_samples_per_second': 3.572, 'train_steps_per_second': 0.232, 'total_flos': 1.15434160128e+17, 'train_loss': 3.0944142708411584, 'epoch': 2.0})

In [33]:
SAVE_DIR = "/content/drive/MyDrive/phowhisper-child-5-7-final"
trainer.model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("Đã lưu xong vào:", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Đã lưu xong vào: /content/drive/MyDrive/phowhisper-child-5-7-final


In [34]:
import torch
from transformers import WhisperProcessor, WhisperForConditionalGeneration

FINAL_DIR = "/content/drive/MyDrive/phowhisper-child-5-7-final"
device = "cuda" if torch.cuda.is_available() else "cpu"

ft_processor = WhisperProcessor.from_pretrained(FINAL_DIR, local_files_only=True)
ft_model = WhisperForConditionalGeneration.from_pretrained(
    FINAL_DIR,
    local_files_only=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
ft_model.eval()
print("Nạp lại model fine-tune thành công trên:", device)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Nạp lại model fine-tune thành công trên: cuda


In [35]:
import torch
from jiwer import wer, cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

FINAL_DIR = "/content/drive/MyDrive/phowhisper-child-5-7-final"
device = "cuda" if torch.cuda.is_available() else "cpu"

ft_processor = WhisperProcessor.from_pretrained(FINAL_DIR, local_files_only=True)
ft_model = WhisperForConditionalGeneration.from_pretrained(
    FINAL_DIR,
    local_files_only=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)
ft_model.eval()
ft_model.generation_config.language = "vi"
ft_model.generation_config.task = "transcribe"
ft_model.generation_config.forced_decoder_ids = None

def transcribe_ft(audio_16k, sampling_rate=16000) -> str:
    inputs = ft_processor(audio_16k, sampling_rate=sampling_rate, return_tensors="pt")
    input_features = inputs.input_features.to(
        device,
        dtype=torch.float16 if device == "cuda" else torch.float32,
    )
    with torch.no_grad():
        pred_ids = ft_model.generate(input_features)
    return ft_processor.batch_decode(pred_ids, skip_special_tokens=True)[0]

refs, hyps = [], []
for sample in eval_samples:
    refs.append(normalize_vi(sample["text"]))
    hyps.append(normalize_vi(transcribe_ft(sample["audio"])))

print(f"Post-finetune WER: {wer(refs, hyps):.4f} ({wer(refs, hyps)*100:.2f}%)")
print(f"Post-finetune CER: {cer(refs, hyps):.4f} ({cer(refs, hyps)*100:.2f}%)")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Post-finetune WER: 0.0000 (0.00%)
Post-finetune CER: 0.0000 (0.00%)


In [36]:
import io, json, os, re
import numpy as np
import librosa
import torch
from gtts import gTTS
from jiwer import wer, cer

# 1. Lấy danh sách từ/cụm từ từ data/curriculum.json
def load_curriculum_items(path="data/curriculum.json", limit=20):
    items = []
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        def walk(obj):
            if len(items) >= limit:
                return
            if isinstance(obj, str):
                s = obj.strip()
                if 1 <= len(s) <= 35 and s not in items:
                    items.append(s)
            elif isinstance(obj, dict):
                for v in obj.values():
                    walk(v)
            elif isinstance(obj, list):
                for x in obj:
                    walk(x)
        walk(data)
    return items or ["làng quê", "nàng tiên", "trâu rừng", "con chim", "bé đọc bài", "mẹ nấu cơm"]

curriculum_texts = load_curriculum_items("data/curriculum.json", limit=20)

# 2. Tạo audio mô phỏng giọng trẻ 5–7 tuổi (tăng pitch + tốc độ + nhiễu nhẹ)
rng = np.random.default_rng(42)
child_eval_samples = []

for txt in curriculum_texts:
    buf = io.BytesIO()
    gTTS(text=txt, lang="vi").write_to_fp(buf)
    buf.seek(0)
    y, _ = librosa.load(buf, sr=16000, mono=True)
    
    # Tăng cao độ +3 nửa cung (giọng cao hơn giống trẻ em)
    y_child = librosa.effects.pitch_shift(y, sr=16000, n_steps=3.0)
    # Thay đổi nhẹ tốc độ nói (0.95x - 1.08x)
    rate = float(rng.uniform(0.95, 1.08))
    y_child = librosa.effects.time_stretch(y_child, rate=rate)
    # Thêm nhiễu môi trường nhẹ
    noise = rng.normal(0, 0.004, size=y_child.shape).astype(np.float32)
    y_child = np.clip(y_child + noise, -1.0, 1.0)
    
    child_eval_samples.append({"audio": y_child, "text": txt})

def normalize_vi(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\sàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def eval_model(model_obj, proc_obj, samples):
    refs, hyps = [], []
    for s in samples:
        inputs = proc_obj(s["audio"], sampling_rate=16000, return_tensors="pt")
        feats = inputs.input_features.to(
            device, dtype=torch.float16 if device == "cuda" else torch.float32
        )
        with torch.no_grad():
            pred_ids = model_obj.generate(feats)
        hyp = proc_obj.batch_decode(pred_ids, skip_special_tokens=True)[0]
        refs.append(normalize_vi(s["text"]))
        hyps.append(normalize_vi(hyp))
    return wer(refs, hyps), cer(refs, hyps), list(zip(refs, hyps))

base_wer, base_cer, _ = eval_model(base_model, processor, child_eval_samples)
ft_wer, ft_cer, ft_pairs = eval_model(ft_model, ft_processor, child_eval_samples)

print(f"Base Model   -> WER: {base_wer*100:.2f}% | CER: {base_cer*100:.2f}%")
print(f"Child Model  -> WER: {ft_wer*100:.2f}% | CER: {ft_cer*100:.2f}%")
for r, h in ft_pairs[:10]:
    print(f"  REF: {r:<18} | HYP: {h}")

Base Model   -> WER: 100.00% | CER: 53.70%
Child Model  -> WER: 85.71% | CER: 46.30%
  REF: làng quê           | HYP: làm quay
  REF: nàng tiên          | HYP: làm tiên
  REF: trâu rừng          | HYP: sâu rơi
  REF: con chim           | HYP: còn chim
  REF: bé đọc bài         | HYP: mẹ học lời
  REF: mẹ nấu cơm         | HYP: lẻ ngơ ngơ


In [37]:
import unicodedata
from difflib import SequenceMatcher

# Bảng tách dấu thanh tiếng Việt từ nguyên âm có dấu
TONE_MAP = {
    "à": ("a", "huyền"), "á": ("a", "sắc"), "ả": ("a", "hỏi"), "ã": ("a", "ngã"), "ạ": ("a", "nặng"),
    "ằ": ("ă", "huyền"), "ắ": ("ă", "sắc"), "ẳ": ("ă", "hỏi"), "ẵ": ("ă", "ngã"), "ặ": ("ă", "nặng"),
    "ầ": ("â", "huyền"), "ấ": ("â", "sắc"), "ẩ": ("â", "hỏi"), "ẫ": ("â", "ngã"), "ậ": ("â", "nặng"),
    "è": ("e", "huyền"), "é": ("e", "sắc"), "ẻ": ("e", "hỏi"), "ẽ": ("e", "ngã"), "ẹ": ("e", "nặng"),
    "ề": ("ê", "huyền"), "ế": ("ê", "sắc"), "ể": ("ê", "hỏi"), "ễ": ("ê", "ngã"), "ệ": ("ê", "nặng"),
    "ì": ("i", "huyền"), "í": ("i", "sắc"), "ỉ": ("i", "hỏi"), "ĩ": ("i", "ngã"), "ị": ("i", "nặng"),
    "ò": ("o", "huyền"), "ó": ("o", "sắc"), "ỏ": ("o", "hỏi"), "õ": ("o", "ngã"), "ọ": ("o", "nặng"),
    "ồ": ("ô", "huyền"), "ố": ("ô", "sắc"), "ổ": ("ô", "hỏi"), "ỗ": ("ô", "ngã"), "ộ": ("ô", "nặng"),
    "ờ": ("ơ", "huyền"), "ớ": ("ơ", "sắc"), "ở": ("ơ", "hỏi"), "ỡ": ("ơ", "ngã"), "ợ": ("ơ", "nặng"),
    "ù": ("u", "huyền"), "ú": ("u", "sắc"), "ủ": ("u", "hỏi"), "ũ": ("u", "ngã"), "ụ": ("u", "nặng"),
    "ừ": ("ư", "huyền"), "ứ": ("ư", "sắc"), "ử": ("ư", "hỏi"), "ữ": ("ư", "ngã"), "ự": ("ư", "nặng"),
    "ỳ": ("y", "huyền"), "ý": ("y", "sắc"), "ỷ": ("y", "hỏi"), "ỹ": ("y", "ngã"), "ỵ": ("y", "nặng"),
}

INITIALS = [
    "ngh", "ng", "nh", "ch", "gh", "gi", "kh", "ph", "qu", "th", "tr",
    "b", "c", "d", "đ", "g", "h", "k", "l", "m", "n", "p", "q", "r", "s", "t", "v", "x"
]

CONFUSION_HINTS = {
    ("l", "n"): "Bé đang nhầm âm 'l' và 'n'. Nhắc bé cong đầu lưỡi chạm lợi trên khi đọc âm 'l'.",
    ("n", "l"): "Bé đang nhầm âm 'n' và 'l'. Nhắc bé hạ lưỡi phẳng và đẩy hơi qua mũi khi đọc âm 'n'.",
    ("tr", "ch"): "Bé đang đọc 'tr' thành 'ch'. Nhắc bé uốn đầu lưỡi lên vòm miệng khi đọc 'tr'.",
    ("s", "x"): "Bé đang đọc 's' thành 'x'. Nhắc bé uốn nhẹ đầu lưỡi khi đọc 's'.",
    ("r", "d"): "Bé đang đọc 'r' thành 'd'. Nhắc bé rung nhẹ đầu lưỡi khi đọc 'r'.",
}

def parse_vi_syllable(syllable: str) -> dict:
    s = unicodedata.normalize("NFC", syllable.strip().lower())
    tone = "ngang"
    base_chars = []
    for ch in s:
        if ch in TONE_MAP:
            base_ch, found_tone = TONE_MAP[ch]
            base_chars.append(base_ch)
            tone = found_tone
        else:
            base_chars.append(ch)
    plain = "".join(base_chars)

    initial = ""
    for ini in INITIALS:
        if plain.startswith(ini):
            # Xử lý riêng "gi" đứng trước nguyên âm (ví dụ: "già" -> đầu "gi", vần "a")
            if ini == "gi" and len(plain) > 2:
                initial = "gi"
            elif ini != "gi":
                initial = ini
            break

    rime = plain[len(initial):]
    return {"raw": s, "initial": initial, "rime": rime, "tone": tone}

def analyze_pronunciation(target_text: str, spoken_text: str) -> dict:
    ref_words = normalize_vi(target_text).split()
    hyp_words = normalize_vi(spoken_text).split()

    matcher = SequenceMatcher(None, ref_words, hyp_words)
    details = []
    correct_count = 0

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            for w in ref_words[i1:i2]:
                correct_count += 1
                details.append({"target": w, "spoken": w, "status": "correct", "errors": []})
        elif tag == "replace":
            span = max(i2 - i1, j2 - j1)
            for k in range(span):
                rw = ref_words[i1 + k] if (i1 + k) < i2 else ""
                hw = hyp_words[j1 + k] if (j1 + k) < j2 else ""
                if not rw:
                    continue
                if not hw:
                    details.append({
                        "target": rw, "spoken": "", "status": "missing",
                        "errors": [f"Bé đọc thiếu tiếng '{rw}'."]
                    })
                    continue

                p_ref = parse_vi_syllable(rw)
                p_hyp = parse_vi_syllable(hw)
                errs = []

                if p_ref["initial"] != p_hyp["initial"]:
                    hint = CONFUSION_HINTS.get(
                        (p_ref["initial"], p_hyp["initial"]),
                        f"Sai âm đầu: cần đọc '{p_ref['initial'] or '(không âm đầu)'}', bé đọc thành '{p_hyp['initial'] or '(không âm đầu)'}'."
                    )
                    errs.append(hint)

                if p_ref["rime"] != p_hyp["rime"]:
                    errs.append(f"Sai phần vần/âm cuối: cần đọc vần '{p_ref['rime']}', bé đọc thành '{p_hyp['rime']}'.")

                if p_ref["tone"] != p_hyp["tone"]:
                    errs.append(f"Sai thanh điệu: cần đọc thanh {p_ref['tone']}, bé đọc thành thanh {p_hyp['tone']}.")

                details.append({
                    "target": rw,
                    "spoken": hw,
                    "status": "incorrect" if errs else "correct",
                    "errors": errs,
                })
        elif tag == "delete":
            for rw in ref_words[i1:i2]:
                details.append({
                    "target": rw, "spoken": "", "status": "missing",
                    "errors": [f"Bé đọc thiếu tiếng '{rw}'."]
                })

    score = round((correct_count / max(len(ref_words), 1)) * 100, 1)
    return {
        "target_text": target_text,
        "spoken_text": spoken_text,
        "score": score,
        "syllable_details": details,
    }

In [38]:
sample = child_eval_samples[0]
recognized = transcribe_ft(sample["audio"])
report = analyze_pronunciation(target_text="làng quê", spoken_text="nàng quê")
print(json.dumps(report, ensure_ascii=False, indent=2))

{
  "target_text": "làng quê",
  "spoken_text": "nàng quê",
  "score": 50.0,
  "syllable_details": [
    {
      "target": "làng",
      "spoken": "nàng",
      "status": "incorrect",
      "errors": [
        "Bé đang nhầm âm 'l' và 'n'. Nhắc bé cong đầu lưỡi chạm lợi trên khi đọc âm 'l'."
      ]
    },
    {
      "target": "quê",
      "spoken": "quê",
      "status": "correct",
      "errors": []
    }
  ]
}


In [39]:
!rm -rf /content/tmp_child_asr
!ls -lh /content/drive/MyDrive/phowhisper-child-5-7-final

total 926M
-rw------- 1 root root 1.3K Nov 25 15:29 config.json
-rw------- 1 root root 3.8K Nov 25 15:29 generation_config.json
-rw------- 1 root root 923M Nov 25 15:29 model.safetensors
-rw------- 1 root root  409 Nov 25 15:29 processor_config.json
-rw------- 1 root root 2.1K Nov 25 15:29 tokenizer_config.json
-rw------- 1 root root 3.5M Nov 25 15:29 tokenizer.json
